# AQG - IndoT5 LoRA Fine-Tuning
Notebook ini digunakan untuk mengeksekusi fine-tuning model generasi pertanyaan (AQG) dari IndoT5 di Google Colab.

In [ ]:
# @title ## 0. Persiapan Awal (Bootstrap)
# Beritahu Colab di mana repo Anda agar folder 'src' dan 'data' terbaca
import os
import sys

PROJECT_URL = "https://github.com/mdrnid/AQG.git"
REPO_NAME = "AQG"

if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone {PROJECT_URL}
    %cd {REPO_NAME}
else:
    %cd /content/{REPO_NAME}

print("Posisi Kerja Sekarang:", os.getcwd())

## 1. Setup Environment & Detect Colab
Script ini akan otomatis menginstal library yang diperlukan.

In [ ]:
import os
import sys
import torch

# Deteksi apakah berjalan di Google Colab
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    print("Detected Google Colab environment.")
    
    # CEK GPU
    if not torch.cuda.is_available():
        print("\n" + "="*50)
        print("CRITICAL WARNING: GPU NOT DETECTED!")
        print("Training di CPU akan sangat lambat (bisa berminggu-minggu).")
        print("Silakan klik: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
        print("="*50 + "\n")
    else:
        print(f"GPU Detected: {torch.cuda.get_device_name(0)}")

    # Update pip dan install dependensi utama
    %pip install -q transformers peft datasets accelerate bitsandbytes evaluate rouge-score sentencepiece
    
    # Mount Google Drive untuk persistensi model/checkpoints
    if not os.path.exists('/content/drive'):
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print(f"Bypass drive mount: {e}")
    else:
        print("Google Drive already mounted.")
else:
    print("Detected Local environment.")

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

## 2. Load Dataset
Menggunakan modul custom data loader dari `src/finetuning`

In [ ]:
from src.finetuning.data_loader import load_aqg_dataset

# Path dataset relatif dari root project
DATASET_PATH = './dataset_aqg/dataset-task-spesifc'

if IN_COLAB and not os.path.exists(DATASET_PATH):
    print(f"Dataset not found at {DATASET_PATH}. Checking in Google Drive...")
    # Contoh jika ditaruh di Drive:
    DATASET_PATH = '/content/drive/MyDrive/AQG_Dataset/dataset-task-spesifc'

print(f"Loading dataset from: {DATASET_PATH}")
dataset, tokenizer = load_aqg_dataset(DATASET_PATH)

print(f"Dataset loaded: {len(dataset['train'])} train samples")
print(f"Dataset validation: {len(dataset['validation'])} samples")

## 3. Setup LoRA Model
Inisialisasi model referensi IndoT5 beserta LoRA adapter-nya

In [ ]:
from src.finetuning.model_setup import setup_model_with_lora

model = setup_model_with_lora(
    model_name="Wikidepia/IndoT5-base",
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.1
)

## 4. Trainer Configuration

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

if IN_COLAB:
    OUTPUT_DIR = "/content/drive/MyDrive/aqg_checkpoints"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
else:
    OUTPUT_DIR = "./checkpoints"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",            
    learning_rate=5e-5,               # Diturunkan untuk stabilitas (Fix NaN)
    per_device_train_batch_size=4,    
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,    
    num_train_epochs=3,               
    weight_decay=0.01,
    save_total_limit=3,               
    predict_with_generate=True,       
    fp16=True,                        
    logging_steps=50,
    save_strategy="epoch"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    data_collator=data_collator
)

## 5. Start Training 🚀

In [ ]:
trainer.train() 

## 6. Save Final Model

In [ ]:
trainer.save_model("./final_model")
print("Model saved successfully!")